# Feature Engineering & Selection

## Project Overview
This notebook performs feature engineering, correlation analysis, and feature selection to prepare the dataset for modeling.

## 1. Import Libraries

In [ ]:
from pandas import DataFrame, Series
import pandas as pd
import numpy as np
import numpy.random as random
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## 2. Load Cleaned Dataset

In [ ]:
# Load the cleaned dataset from previous notebook
auto = pd.read_csv("data/processed/auto_clean.csv")

print(f"Dataset shape: {auto.shape}")
print(f"\nFirst 5 rows:")
auto.head()

In [ ]:
# Check data types
auto.dtypes.value_counts()

## 3. Convert Boolean Columns to Integers

In [ ]:
# Convert boolean columns to 0/1 integers for better correlation analysis
bool_cols = auto.select_dtypes(include='bool').columns
print(f"Found {len(bool_cols)} boolean columns: {bool_cols.tolist()}")

auto[bool_cols] = auto[bool_cols].astype(int)

print("\nBoolean columns converted to integers (0/1)")
auto[bool_cols].head()

## 4. Feature Engineering

Creating new features that capture underlying relationships:

In [ ]:
# Create engineered features
auto['horsepower_per_kg'] = auto['horsepower'] / auto['curb-weight']
auto['engine_per_kg'] = auto['engine-size'] / auto['curb-weight']
auto['mpg_avg'] = (auto['city-mpg'] + auto['highway-mpg']) / 2
auto['footprint'] = auto['length'] * auto['width']

print("New features created:")
print("- horsepower_per_kg: Power-to-weight ratio")
print("- engine_per_kg: Engine size per unit weight")
print("- mpg_avg: Average fuel efficiency")
print("- footprint: Car's length × width")

print(f"\nDataset shape after feature engineering: {auto.shape}")

In [ ]:
# Preview the new features
auto[['horsepower', 'curb-weight', 'horsepower_per_kg', 
      'engine-size', 'engine_per_kg', 
      'city-mpg', 'highway-mpg', 'mpg_avg',
      'length', 'width', 'footprint', 'price']].head()

## 5. Correlation Analysis with Price

In [ ]:
# Check correlations of all features with price
print("=" * 60)
print("CORRELATION WITH PRICE (sorted ascending)")
print("=" * 60)
correlations = auto.corr(numeric_only=True)['price'].sort_values()
print(correlations)

In [ ]:
# Visualize top and bottom correlations with price
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 10 positive correlations
top_corr = correlations.nlargest(11)[1:]  # skip price itself
axes[0].barh(range(len(top_corr)), top_corr.values)
axes[0].set_yticks(range(len(top_corr)))
axes[0].set_yticklabels(top_corr.index)
axes[0].set_xlabel('Correlation with Price')
axes[0].set_title('Top 10 Features Most Positively Correlated with Price')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Bottom 10 negative correlations
bottom_corr = correlations.nsmallest(10)
axes[1].barh(range(len(bottom_corr)), bottom_corr.values)
axes[1].set_yticks(range(len(bottom_corr)))
axes[1].set_yticklabels(bottom_corr.index)
axes[1].set_xlabel('Correlation with Price')
axes[1].set_title('Top 10 Features Most Negatively Correlated with Price')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 6. Multicollinearity Check

Checking if any two features are highly correlated with each other (> 0.9)

In [ ]:
# Calculate correlation matrix
corr_matrix = auto.corr(numeric_only=True)

# Find pairs with absolute correlation above 0.9 (excluding self-correlation)
high_corr = (corr_matrix.abs() > 0.9) & (corr_matrix != 1.0)
cols = high_corr.any()

print("=" * 60)
print("FEATURES WITH HIGH MULTICOLLINEARITY (corr > 0.9)")
print("=" * 60)

if cols.any():
    high_corr_features = corr_matrix[cols.index[cols]]
    print(high_corr_features)
    
    # Show specific high correlation pairs
    print("\n" + "=" * 60)
    print("SPECIFIC HIGH CORRELATION PAIRS (>0.9):")
    print("=" * 60)
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.9:
                print(f"{corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {corr_val:.3f}")
else:
    print("No features with correlation > 0.9 found")

In [ ]:
# Visualize correlation heatmap (focus on highly correlated blocks)
plt.figure(figsize=(12, 10))

# Get columns that have high correlations
high_corr_cols = cols[cols].index.tolist()
if high_corr_cols:
    # Add price and engineered features to see full picture
    cols_to_plot = list(set(high_corr_cols + ['price', 'horsepower_per_kg', 'engine_per_kg', 'mpg_avg', 'footprint']))
    mask = np.triu(np.ones_like(corr_matrix.loc[cols_to_plot, cols_to_plot], dtype=bool))
    sns.heatmap(corr_matrix.loc[cols_to_plot, cols_to_plot], 
                annot=True, 
                fmt='.2f', 
                cmap='RdBu_r',
                center=0,
                mask=mask,
                square=True,
                linewidths=0.5)
    plt.title('Correlation Heatmap (Focus on Highly Correlated Features)')
    plt.tight_layout()
    plt.show()
else:
    print("No high correlation clusters to visualize")

## 7. Feature Selection

Dropping features based on:
1. **Near-zero correlation with price** (irrelevant features)
2. **Multicollinearity** (redundant features)

In [ ]:
# Features to drop
columns_to_drop = [
    # Near-zero correlation to price
    'fuel-system_mfi',
    'engine-type_rotor',
    'num-of-cylinders_two',
    'fuel-system_4bbl',
    'fuel-system_spfi',
    'body-style_wagon',
    'num-of-doors_two',
    'engine-type_ohcf',
    'make_mercury',
    'make_renault',
    'make_isuzu',
    'drive-wheels_fwd',
    'engine-type_l',
    
    # Multicollinearity (replaced by engineered features)
    'city-mpg',           # replaced by mpg_avg
    'highway-mpg',        # replaced by mpg_avg
    'length',             # replaced by footprint
    'fuel-type_gas',      # inverse of fuel-system_idi
    'compression-ratio',  # ~same as fuel-system_idi
    'width',              # captured by footprint
    'wheel-base',         # captured by footprint
]

# Only drop columns that exist in the dataframe
columns_to_drop_existing = [col for col in columns_to_drop if col in auto.columns]

print(f"Dropping {len(columns_to_drop_existing)} columns:")
for col in columns_to_drop_existing:
    print(f"  - {col}")

auto.drop(columns=columns_to_drop_existing, inplace=True)

print(f"\nDataset shape after feature selection: {auto.shape}")

In [ ]:
# Verify the remaining features
print("=" * 60)
print("REMAINING FEATURES")
print("=" * 60)
remaining_features = auto.columns.tolist()
for i, feature in enumerate(remaining_features, 1):
    print(f"{i:2d}. {feature}")

In [ ]:
# Check correlations again after feature selection
print("=" * 60)
print("CORRELATION WITH PRICE (after feature selection)")
print("=" * 60)
final_correlations = auto.corr(numeric_only=True)['price'].sort_values()
print(final_correlations)

In [ ]:
# Final multicollinearity check
final_corr_matrix = auto.corr(numeric_only=True)
final_high_corr = (final_corr_matrix.abs() > 0.9) & (final_corr_matrix != 1.0)

if final_high_corr.any().any():
    print("WARNING: High correlations still exist (>0.9):")
    for i in range(len(final_corr_matrix.columns)):
        for j in range(i+1, len(final_corr_matrix.columns)):
            corr_val = final_corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.9:
                print(f"  {final_corr_matrix.columns[i]} ↔ {final_corr_matrix.columns[j]}: {corr_val:.3f}")
else:
    print("✓ No high correlations (>0.9) remain - multicollinearity resolved")

## 8. Save Processed Dataset

In [ ]:
# Save the feature-engineered dataset
auto.to_csv('data/processed/auto_features.csv', index=False)
print("Dataset saved as 'data/processed/auto_features.csv'")
print(f"Final dataset shape: {auto.shape}")

In [ ]:
# Quick sanity check - preview final dataset
print("\n" + "=" * 60)
print("FINAL DATASET PREVIEW")
print("=" * 60)
auto.head()

## 9. Summary Statistics

In [ ]:
# Summary of the feature engineering process
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print("\nNew features created:")
print("  • horsepower_per_kg = horsepower / curb-weight")
print("  • engine_per_kg = engine-size / curb-weight")
print("  • mpg_avg = (city-mpg + highway-mpg) / 2")
print("  • footprint = length × width")

print("\nFeatures dropped (low correlation with price):")
print("  • Rare make columns (mercury, renault, isuzu)")
print("  • Rare fuel system types (mfi, 4bbl, spfi)")
print("  • Rare engine types (rotor, ohcf, l)")
print("  • num-of-cylinders_two, body-style_wagon")
print("  • num-of-doors_two, drive-wheels_fwd")

print("\nFeatures dropped (multicollinearity):")
print("  • city-mpg, highway-mpg → replaced by mpg_avg")
print("  • length, width, wheel-base → captured by footprint")
print("  • fuel-type_gas → inverse of fuel-system_idi")
print("  • compression-ratio → similar to fuel-system_idi")

print(f"\nFinal feature count: {auto.shape[1]}")
print(f"Final sample count: {auto.shape[0]}")
print("\n✓ Feature engineering and selection complete!")